# Job Application Tracker

This project is about getting linkedin jobs post links, researching about the company and creating a curated cover letter as well as a tailored resume. You can get the linkedin job post url by going to linkedin and create a curated job list and use the same url. Put your linkedin profile pdf and a summary of the roles you are looking for in a me folder that is in the same location as this notebook.

### NB:This notebook has been modified to use pushover instead of emails.


In [ ]:
# Imports
import os
import asyncio
import requests

from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    WebSearchTool,
)
from agents.model_settings import ModelSettings
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from pydantic import BaseModel, Field
from typing import Dict
from uuid import uuid4

load_dotenv(override=True)

In [ ]:
# globals
test_url = "https://www.linkedin.com/jobs/search/?currentJobId=4386842922&f_F=it%2Ceng&f_I=4%2C96%2C1594%2C6&f_JT=F%2CP%2CC&f_T=9%2C25169%2C25201%2C30128%2C39&f_WT=2&mcid=7394772845974863873&origin=JOB_SEARCH_PAGE_JOB_FILTER&sortBy=R&spellCorrectionEnabled=true&trk=li_LOL_SPIN_global_careers_jobsgtm_ja1_acq_Dec2020_spinv2"
linkedin_job_url = os.getenv("LINKEDIN_JOB_URL", test_url)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

username: str | None = os.getenv("USERNAME")

# initialize clients
openai = OpenAI()

# set model
open_ai_model = "gpt-4o-mini"

In [ ]:
# get the data from linkedin job postings
NO_OF_POSTINGS = 10


class JobPosting(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid4()))
    title: str
    company_name: str
    company_url: str
    location: str
    salary_range: str
    job_description: str
    job_requirements: str
    technologies_needed: str
    must_have_skills: str
    link_to_job_posting: str
    job_posting_date: str


class JobPostingList(BaseModel):
    job_postings: list[JobPosting]


LISTING_INSTRUCTIONS = f"""
You are a job listing assistant helping {username} find relevant job opportunities.

## Objective
Given the job posting url, you will extract the job posting details and return them in a structured format.

## Requirements
- Return EXACTLY {NO_OF_POSTINGS} job postings.
- Prioritize recency (most recent postings first).
- Ensure jobs are relevant to the {username}'s profile.

## Data Quality Rules
For each job posting:
- title: Clear and accurate job title.
- company_name: Company name only.
- company_url: Valid company page URL (if available).
- location: City/Country or "Remote" if applicable.
- salary_range: Extract if available, otherwise "Not specified".
- job_description: Concise summary (3-5 sentences, not full copy).
- job_requirements: Key responsibilities (bullet-style but as text).
- technologies_needed: Extract explicit tools/tech (e.g. Python, React, AWS).
- must_have_skills: Core required skills only (not nice-to-haves).
- link_to_job_posting: Direct job URL (must be valid).
- job_posting_date: Relative or exact (e.g. "2 days ago" or "2026-03-10").

## Constraints
- Do NOT hallucinate links or companies.
- If data is missing, use "Not specified".
- Avoid duplication.
- Keep descriptions concise and structured.

## Output
- Return ONLY structured data matching JobPostingList schema.
- No explanations, no extra text.
- Return exactly {NO_OF_POSTINGS} job postings.
"""

listing_message = f"Fetch job postings from LinkedIn using the link: {linkedin_job_url}"

listing_agent = Agent(
    name="Listing Agent",
    instructions=LISTING_INSTRUCTIONS,
    model=open_ai_model,
    output_type=JobPostingList,
)

In [ ]:
# function tool to send emails


@function_tool
def send_push(subject: str, html_body: str) -> Dict[str, str]:
    """Send a push notification with the given subject and body"""
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": html_body,
        "title": subject,
        "html": 1,
    }
    response = requests.post(pushover_url, data=payload)
    return {"success": True} if response.status_code == 200 else {"success": False}


EMAIL_INSTRUCTIONS = f"""
You are an Email Agent responsible for sending clean, professional HTML emails containing curated job listings using the provided send_push tool.

## Core Behavior
- Always send exactly ONE email using the tool.
- Always produce well-structured, visually clean HTML.
- Do NOT include plain text — HTML only.

## Objective
Send a curated list of job posting links in a clear, professional, and easy-to-read format.

## Email Structure

### Subject
"Curated Job Opportunities for {username}"

### Body
- Start with a short, professional intro paragraph.
- Present job listings in a clean, scannable format.

For EACH job:
- Job Title (use <h2>)
- Company Name
- Location (or "Not specified")
- Clickable link to the job posting
- Optional: 1-2 line concise summary

- Ensure clear spacing between job entries.

## Formatting Rules
- Use semantic HTML: <h2>, <p>, <ul>, <li>, <a>.
- Ensure all links are clickable: <a href="...">View Job</a>.
- Keep tone professional and concise.
- Avoid verbosity or unnecessary text.

## Constraints
- Do NOT hallucinate job details or links.
- If any field is missing, use "Not specified".
- Avoid duplication of job listings.

## Output
Call the send_push tool with:
- subject
- html_body (clean HTML)

Do not output anything else.
"""
email_message = "Create an email for the following job postings"
email_agent = Agent(
    name="Email agent",
    instructions=EMAIL_INSTRUCTIONS,
    tools=[send_push],
    model=open_ai_model,
    handoff_description="Convert an email to HTML and send it",
)

In [ ]:
# review the job postings
# function tools to get the user's linkedin profile and summary
@function_tool
def get_linkedin_profile():
    """Get the linkedin profile of the user placed in ./me/linkedin.pdf"""
    with open("./me/linkedin.pdf", "rb") as pdf_file:
        pdf_reader = PdfReader(pdf_file)
        text = ""
        for page in pdf_reader.pages:
            text += page.extract_text()
    return text


@function_tool
def get_user_summary():
    """Get the summary of the user placed in ./me/summary.txt"""
    with open("./me/summary.txt", "r", encoding="utf-8") as file:
        return file.read()


# Evaluation
class Evaluation(BaseModel):
    """Evaluation of the job posting"""

    is_acceptable: bool
    feedback: str
    job_posting_id: str


EVALUATOR_INSTRUCTIONS = f"""
You are a Job Fit Evaluation Agent.

## Objective
Given a job posting link, determine whether the role is a strong fit for {username} based on their experience, skills, and tech stack.

## Required Actions
1. Retrieve the {username}'s background:
   - Use tools to get LinkedIn profile and summary.
2. Analyze the job posting:
   - Extract key responsibilities, required skills, and technologies from the provided link.

## Evaluation Criteria

### 1. Skills Match
- Do the required skills align with {username}'s actual experience?
- Focus on MUST-HAVE skills, not nice-to-haves.

### 2. Tech Stack Alignment
- Compare required technologies with {username}'s known stack.
- Strong match = majority overlap.

### 3. Experience Level
- Is the role seniority appropriate (junior/mid/senior)?

### 4. Domain Relevance
- Is the role aligned with {username}'s focus (e.g. software engineering, ML infrastructure, frontend, etc.)?

## Decision Rules
- `is_acceptable = true` ONLY if:
  - There is a strong overlap in core skills AND tech stack
  - AND the role is at an appropriate experience level
- Otherwise, `is_acceptable = false`

## Feedback Guidelines
- Be concise and specific.
- Highlight:
  - Matching strengths
  - Missing critical skills (if any)
  - Overall fit reasoning

## Constraints
- Do NOT hallucinate user skills.
- Base decisions strictly on retrieved profile + summary.
- If job details are unclear or insufficient, mark as not acceptable.

## Output
Return ONLY:
- is_acceptable (boolean)
- feedback (clear, actionable reasoning)
- job_posting_id (str) the actual id of the job posting
"""

evaluation_agent = Agent(
    name="Evaluator",
    instructions=EVALUATOR_INSTRUCTIONS,
    model=open_ai_model,
    output_type=Evaluation,
    tools=[
        get_linkedin_profile,
        get_user_summary,
        WebSearchTool(search_context_size="low"),
    ],
    model_settings=ModelSettings(tool_choice="required"),
)


In [ ]:
@function_tool
async def evaluate_job_postings(jobs_list: JobPostingList):
    """Evaluate all job postings using the evaluator agent"""
    tasks = [
        asyncio.create_task(evaluate_job_post(job_post))
        for job_post in jobs_list.job_postings
    ]
    return await asyncio.gather(*tasks)


async def evaluate_job_post(job_post: JobPosting):
    """Evaluate a single job posting using the evaluator agent"""
    evaluation_message = f"Evaluate the job posting: {job_post.link_to_job_posting}"
    evaluation_result = await Runner.run(evaluation_agent, evaluation_message)
    return evaluation_result.final_output


@function_tool
async def get_relevant_postings(
    evaluations: list[Evaluation], jobs_list: JobPostingList
):
    """Get the relevant postings from the evaluations"""
    return [
        job
        for job in jobs_list.job_postings
        if job.id
        in [
            evaluation.job_posting_id
            for evaluation in evaluations
            if evaluation.is_acceptable
        ]
    ]


In [ ]:
LISTING_MANAGER_INSTRUCTIONS = f"""
You are a Job Listing Manager responsible for orchestrating the end-to-end job discovery and delivery process.

## Objective
Identify high-quality job opportunities for {username} and ensure only relevant, well-matched roles are sent to the Email Agent.

## Workflow

### 1. Retrieve Job Listings
- Call the Listing Agent to fetch job postings.
- Ensure you receive a structured list of jobs.

### 2. Evaluate Each Job
- For EACH job posting:
  - Pass the job link to the Evaluation Agent.
  - Collect the evaluation result (`is_acceptable`, `feedback`).

### 3. Filter Jobs
- Keep ONLY jobs where `is_acceptable = true`.
- Discard all others.

### 4. Prepare Output
- If there are accepted jobs:
  - Compile them into a clean, structured list.
  - Ensure each job includes:
    - title
    - company_name
    - location
    - salary_range
    - job_description
    - job_requirements
    - technologies_needed
    - must_have_skills
    - link_to_job_posting
    - short summary (if available)
- If NO jobs are accepted:
  - Do NOT call the Email Agent.
  - Return a clear message indicating no suitable jobs were found.

### 5. Send to Email Agent
- Pass ONLY the filtered, relevant job postings to the Email Agent.
- Do NOT modify job links.
- Do NOT include rejected jobs.

## Constraints
- Do NOT hallucinate job data.
- Do NOT skip evaluation.
- Ensure strict filtering (quality over quantity).
- Avoid duplicates.

## Output Behavior
- If jobs exist → call Email Agent with curated listings.
- If no jobs → return a short message: "No suitable job matches found."

Do not output anything else.
"""

listing_tool = listing_agent.as_tool(
    tool_name="listing_tool", tool_description=listing_message
)

listing_manager = Agent(
    name="Listing Manager",
    instructions=LISTING_MANAGER_INSTRUCTIONS,
    tools=[listing_tool, evaluate_job_postings, get_relevant_postings],
    handoffs=[email_agent],
    model=open_ai_model,
)

manager_message = f"Send out an email to {username} with the job postings"

with trace("Automated LinkedIN Job Postings Hunter"):
    result = await Runner.run(listing_manager, manager_message)
